In [1]:
# synthetic_data_generator_v2.py
"""
Extended synthetic dataset generator.
Encodes real-world relationships:
  - Older/higher-km cars cost more to repair (parts scarcity)
  - Premium segments have higher parts cost
  - Depreciation affects IDV (Insured Declared Value)
  - Multiple damages interact (non-linearly)
"""

import random
import numpy as np
import pandas as pd
from pathlib import Path

random.seed(42)
np.random.seed(42)

# ── Domain Knowledge Tables ───────────────────────────────────────────────────

CAR_SEGMENTS = {
    "hatchback":  {"base_multiplier": 1.0,  "examples": ["Alto", "Swift", "i10", "Polo"]},
    "sedan":      {"base_multiplier": 1.3,  "examples": ["Dzire", "City", "Verna", "Ciaz"]},
    "suv_compact":{"base_multiplier": 1.5,  "examples": ["Brezza", "Venue", "Sonet", "Tata Nexon"]},
    "suv_full":   {"base_multiplier": 2.0,  "examples": ["Creta", "Seltos", "Fortuner", "XUV700"]},
    "luxury":     {"base_multiplier": 3.5,  "examples": ["BMW 3", "Audi A4", "Mercedes C", "Volvo S60"]},
}

BASE_COST_INR = {
    "dent":          (3000,  10000),
    "scratch":       (1500,   6000),
    "crack":         (5000,  15000),
    "glass shatter": (8000,  25000),
    "tire flat":     (2000,   8000),
    "lamp broken":   (3500,  12000),
}

CLASS_NAMES = ["dent", "scratch", "crack", "glass shatter", "tire flat", "lamp broken"]

# IDV depreciation schedule (as per standard Indian motor insurance)
DEPRECIATION_SCHEDULE = {
    0: 0.05,   # <6 months: 5%
    1: 0.15,   # 1 year: 15%
    2: 0.20,
    3: 0.30,
    4: 0.40,
    5: 0.50,
    6: 0.60,   # 5+ years: 60%+
}

def get_depreciation_factor(car_age_years: int) -> float:
    age = min(car_age_years, 6)
    return 1.0 - DEPRECIATION_SCHEDULE[age]


def get_km_multiplier(km_driven: int) -> float:
    """Higher km = higher repair cost (worn parts, harder repairs)."""
    if km_driven < 20000:   return 0.90
    if km_driven < 50000:   return 1.00
    if km_driven < 80000:   return 1.10
    if km_driven < 120000:  return 1.20
    return 1.35


def get_age_parts_multiplier(car_age: int) -> float:
    """Older cars: parts scarcity + compatibility issues."""
    if car_age <= 2:  return 1.00
    if car_age <= 4:  return 1.08
    if car_age <= 6:  return 1.18
    if car_age <= 9:  return 1.30
    return 1.45


def estimate_idv(purchase_price: float, car_age: int) -> float:
    """IDV = Insured Declared Value = purchase_price * (1 - depreciation)."""
    dep = get_depreciation_factor(car_age)
    return purchase_price * dep


def generate_one_scenario(idx: int) -> dict:
    # ── Vehicle metadata ──────────────────────────────────────────────────────
    segment = random.choice(list(CAR_SEGMENTS.keys()))
    seg_info = CAR_SEGMENTS[segment]
    seg_mult = seg_info["base_multiplier"]

    # Realistic purchase price per segment (INR)
    price_ranges = {
        "hatchback":   (400_000,  700_000),
        "sedan":       (700_000, 1_200_000),
        "suv_compact": (900_000, 1_500_000),
        "suv_full":    (1_500_000, 3_000_000),
        "luxury":      (3_000_000, 8_000_000),
    }
    purchase_price = random.randint(*price_ranges[segment])

    car_age = random.choices(
        [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
        weights=[5, 10, 12, 12, 11, 10, 9, 8, 8, 8, 7]
    )[0]

    # km_driven correlates with age but has variance
    avg_km_per_year = random.randint(8000, 18000)
    km_driven = int(car_age * avg_km_per_year * random.uniform(0.7, 1.3))
    km_driven = max(0, min(km_driven, 250_000))

    idv = estimate_idv(purchase_price, car_age)

    # ── Damage detections (simulated YOLO output) ─────────────────────────────
    n_damages = random.choices([1, 2, 3, 4, 5], weights=[30, 28, 20, 13, 9])[0]
    detections = []
    for _ in range(n_damages):
        cls  = random.choice(CLASS_NAMES)
        conf = random.uniform(0.52, 0.99)
        x1   = random.randint(0, 500)
        y1   = random.randint(0, 500)
        x2   = random.randint(x1 + 20, min(x1 + 300, 640))
        y2   = random.randint(y1 + 20, min(y1 + 300, 640))
        detections.append({"class": cls, "confidence": conf,
                            "bbox": [x1, y1, x2, y2]})

    # ── Cost computation ──────────────────────────────────────────────────────
    img_area      = 640 * 640
    km_mult       = get_km_multiplier(km_driven)
    age_parts_mul = get_age_parts_multiplier(car_age)

    total_repair = 0
    class_counts = {c: 0 for c in CLASS_NAMES}
    areas, confs, sev_scores = [], [], []
    sev_map = {"small": 1, "medium": 2, "large": 3}

    for det in detections:
        cls = det["class"]
        conf = det["confidence"]
        x1, y1, x2, y2 = det["bbox"]
        area_frac = (x2 - x1) * (y2 - y1) / img_area

        if area_frac < 0.05:   sev, sev_mult = "small",  1.0
        elif area_frac < 0.15: sev, sev_mult = "medium", 1.5
        else:                  sev, sev_mult = "large",  2.2

        base_min, base_max = BASE_COST_INR.get(cls, (2000, 8000))
        base_cost = (base_min + base_max) / 2

        item_cost = (
            base_cost
            * sev_mult
            * seg_mult           # premium car = expensive parts
            * km_mult            # high km = harder repairs
            * age_parts_mul      # old car = scarce parts
            * (0.8 + conf * 0.2) # confidence adjustment
        )

        total_repair += item_cost
        class_counts[cls] += 1
        areas.append(area_frac)
        confs.append(conf)
        sev_scores.append(sev_map[sev])

    # Multi-damage penalty
    if n_damages > 1:
        total_repair *= (1 + (n_damages - 1) * 0.10)

    # IDV-cap logic: claim can't exceed IDV
    total_repair = min(total_repair, idv * 0.85)

    # Labour cost (20-30% on top of parts)
    labour_pct = random.uniform(0.20, 0.30)
    total_repair *= (1 + labour_pct)

    # Add realistic noise (±15%)
    noise = random.uniform(0.85, 1.15)
    claim_amount = round(total_repair * noise)

    # ── Build feature row ─────────────────────────────────────────────────────
    row = {
        # Vehicle features
        "car_age":             car_age,
        "km_driven":           km_driven,
        "km_per_year":         km_driven / max(car_age, 1),
        "purchase_price":      purchase_price,
        "idv":                 round(idv),
        "is_luxury":           int(segment == "luxury"),
        "is_premium":          int(segment in ["suv_full", "luxury"]),
        "segment_multiplier":  seg_mult,
        "depreciation_factor": get_depreciation_factor(car_age),
        "is_high_mileage":     int(km_driven > 80_000),
        "is_old_car":          int(car_age > 5),

        # Damage features (from YOLO)
        "damage_count":        n_damages,
        "total_area_frac":     sum(areas),
        "max_area_frac":       max(areas),
        "avg_area_frac":       sum(areas) / len(areas),
        "avg_confidence":      sum(confs) / len(confs),
        "min_confidence":      min(confs),
        "max_severity_score":  max(sev_scores),
        "avg_severity_score":  sum(sev_scores) / len(sev_scores),

        # One-hot damage type counts
        **{f"count_{c.replace(' ', '_')}": class_counts[c] for c in CLASS_NAMES},

        # Derived interaction features
        "has_glass_shatter":   int(class_counts["glass shatter"] > 0),
        "has_crack":           int(class_counts["crack"] > 0),
        "has_lamp_broken":     int(class_counts["lamp broken"] > 0),
        "damage_value_ratio":  sum(areas) * purchase_price / 100_000,

        # Target
        "claim_amount":        claim_amount,
    }
    return row


def generate_dataset(n: int = 10_000) -> pd.DataFrame:
    rows = [generate_one_scenario(i) for i in range(n)]
    df = pd.DataFrame(rows)
    print(f"Generated {len(df)} samples")
    print(f"Claim range: ₹{df.claim_amount.min():,.0f} — ₹{df.claim_amount.max():,.0f}")
    print(f"Claim mean:  ₹{df.claim_amount.mean():,.0f}")
    return df


if __name__ == "__main__":
    df = generate_dataset(10_000)
    df.to_csv("extended_training_data.csv", index=False)
    print("Saved → extended_training_data.csv")

Generated 10000 samples
Claim range: ₹3,228 — ₹897,595
Claim mean:  ₹84,993
Saved → extended_training_data.csv


In [ ]:
# %pip install joblib 

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# %pip install scikit-learn

  Using cached scikit_learn-1.8.0-cp311-cp311-win_amd64.whl.metadata (11 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.8.0-cp311-cp311-win_amd64.whl (8.1 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# %pip install xgboost

  Using cached xgboost-3.2.0-py3-none-win_amd64.whl.metadata (2.1 kB)
Using cached xgboost-3.2.0-py3-none-win_amd64.whl (101.7 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# %pip install pathlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# train_extended_model.py
"""
Train XGBoost model on extended features (damage + vehicle).
Outputs: extended_cost_model.pkl, extended_model_metadata.json
"""

import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor
from pathlib import Path

# ── Load data ─────────────────────────────────────────────────────────────────
df = pd.read_csv("extended_training_data.csv")

TARGET = "claim_amount"
# Drop IDV to prevent data leakage in deployment (user won't enter purchase price)
# Keep segment_multiplier as a proxy (derived from car model dropdown)
FEATURE_COLS = [c for c in df.columns if c != TARGET]

X = df[FEATURE_COLS]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42
)

# ── Scale ─────────────────────────────────────────────────────────────────────
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# ── Train XGBoost ─────────────────────────────────────────────────────────────
model = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
)
model.fit(
    X_train_s, y_train,
    eval_set=[(X_test_s, y_test)],
    verbose=50,
)

# ── Evaluate ──────────────────────────────────────────────────────────────────
y_pred = model.predict(X_test_s)
mae    = mean_absolute_error(y_test, y_pred)
r2     = r2_score(y_test, y_pred)
mape   = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print(f"\n{'='*45}")
print(f"  Extended Model Results")
print(f"{'='*45}")
print(f"  MAE  : ₹{mae:,.0f}")
print(f"  R²   : {r2:.4f}")
print(f"  MAPE : {mape:.2f}%")

# Feature importance
feat_imp = pd.Series(model.feature_importances_, index=FEATURE_COLS)
print(f"\n  Top 10 Features:")
print(feat_imp.sort_values(ascending=False).head(10).to_string())

# ── Save ──────────────────────────────────────────────────────────────────────
Path("models").mkdir(exist_ok=True)
joblib.dump(model,  "models/extended_cost_model.pkl")
joblib.dump(scaler, "models/extended_scaler.pkl")

metadata = {
    "feature_names":   FEATURE_COLS,
    "class_names":     ["dent", "scratch", "crack", "glass shatter", "tire flat", "lamp broken"],
    "metrics":         {"MAE": mae, "R2": r2, "MAPE": mape},
    "model_version":   "2.0-extended",
}
with open("models/extended_model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("\n✅ Saved to models/")

[0]	validation_0-rmse:83378.11287
[50]	validation_0-rmse:23812.05559
[100]	validation_0-rmse:18501.31009
[150]	validation_0-rmse:17252.44473
[200]	validation_0-rmse:16917.97516
[250]	validation_0-rmse:16770.34357
[300]	validation_0-rmse:16673.68652
[350]	validation_0-rmse:16631.84714
[400]	validation_0-rmse:16612.12773
[450]	validation_0-rmse:16560.25328
[499]	validation_0-rmse:16544.94621

  Extended Model Results
  MAE  : ₹8,685
  R²   : 0.9638
  MAPE : 10.83%

  Top 10 Features:
is_luxury              0.327505
damage_count           0.170059
damage_value_ratio     0.125256
count_glass_shatter    0.065161
has_crack              0.045260
depreciation_factor    0.039319
has_glass_shatter      0.031178
car_age                0.028696
count_crack            0.024375
km_driven              0.021414

✅ Saved to models/


In [ ]:
# extended_predictor.py
"""
Full prediction pipeline combining:
  - YOLO damage detection (from image)
  - Vehicle metadata (from user input)
→ final_claim_amount (INR)
"""

import json
import numpy as np
import joblib
from pathlib import Path
from ultralytics import YOLO

# ── Car segment lookup (maps user-entered model → segment) ────────────────────
CAR_SEGMENT_MAP = {
    # hatchback
    "alto": "hatchback", "swift": "hatchback", "wagon r": "hatchback",
    "i10": "hatchback", "i20": "hatchback", "polo": "hatchback",
    "baleno": "hatchback", "tata tiago": "hatchback",
    # sedan
    "dzire": "sedan", "honda city": "sedan", "verna": "sedan",
    "ciaz": "sedan", "tata tigor": "sedan",
    # compact suv
    "brezza": "suv_compact", "venue": "suv_compact", "sonet": "suv_compact",
    "tata nexon": "suv_compact", "magnite": "suv_compact",
    # full suv
    "creta": "suv_full", "seltos": "suv_full", "tata harrier": "suv_full",
    "xuv700": "suv_full", "fortuner": "suv_full", "endeavour": "suv_full",
    # luxury
    "bmw": "luxury", "audi": "luxury", "mercedes": "luxury",
    "volvo": "luxury", "jaguar": "luxury",
}

SEGMENT_MULTIPLIERS = {
    "hatchback": 1.0, "sedan": 1.3, "suv_compact": 1.5,
    "suv_full": 2.0,  "luxury": 3.5,
}

DEPRECIATION = {0: 0.05, 1: 0.15, 2: 0.20, 3: 0.30, 4: 0.40, 5: 0.50, 6: 0.60}

BASE_COST_INR = {
    "dent": (3000, 10000), "scratch": (1500, 6000), "crack": (5000, 15000),
    "glass shatter": (8000, 25000), "tire flat": (2000, 8000), "lamp broken": (3500, 12000),
}
CLASS_NAMES = ["dent", "scratch", "crack", "glass shatter", "tire flat", "lamp broken"]


def resolve_segment(car_model: str) -> str:
    """Map user-entered car model string → segment key."""
    model_lower = car_model.lower().strip()
    for key, segment in CAR_SEGMENT_MAP.items():
        if key in model_lower:
            return segment
    return "sedan"  # safe default


def get_depreciation_factor(car_age: int) -> float:
    age = min(car_age, 6)
    return 1.0 - DEPRECIATION[age]


class ExtendedPredictor:

    def __init__(
        self,
        yolo_path: str,
        model_path: str = "models/extended_cost_model.pkl",
        scaler_path: str = "models/extended_scaler.pkl",
        metadata_path: str = "models/extended_model_metadata.json",
    ):
        self.yolo   = YOLO(yolo_path)
        self.model  = joblib.load(model_path)
        self.scaler = joblib.load(scaler_path)
        with open(metadata_path) as f:
            self.meta = json.load(f)
        self.feature_names = self.meta["feature_names"]

    def _run_yolo(self, image_path: str):
        results  = self.yolo(image_path, verbose=False)[0]
        img_h, img_w = results.orig_shape
        detections = []
        for box in results.boxes:
            cls_id = int(box.cls.item())
            conf   = float(box.conf.item())
            x1, y1, x2, y2 = [float(v) for v in box.xyxy[0]]
            detections.append({
                "class":      CLASS_NAMES[cls_id],
                "class_id":   cls_id,
                "confidence": round(conf, 3),
                "bbox":       [round(x1), round(y1), round(x2), round(y2)],
            })
        return detections, img_w, img_h

    def _build_features(
        self,
        detections: list,
        img_w: int,
        img_h: int,
        car_model: str,
        car_age: int,
        km_driven: int,
    ) -> np.ndarray:
        img_area = img_w * img_h
        segment  = resolve_segment(car_model)
        seg_mult = SEGMENT_MULTIPLIERS[segment]
        dep_fac  = get_depreciation_factor(car_age)

        # Approximate IDV from segment (no exact price needed)
        segment_avg_price = {
            "hatchback": 550_000, "sedan": 900_000, "suv_compact": 1_200_000,
            "suv_full": 2_000_000, "luxury": 5_000_000,
        }
        est_purchase_price = segment_avg_price[segment]
        idv = est_purchase_price * dep_fac

        class_counts = {c: 0 for c in CLASS_NAMES}
        areas, confs, sev_scores = [], [], []
        sev_map = {"small": 1, "medium": 2, "large": 3}

        for det in detections:
            area_frac = (
                (det["bbox"][2] - det["bbox"][0]) *
                (det["bbox"][3] - det["bbox"][1])
            ) / img_area
            if area_frac < 0.05:   sev = "small"
            elif area_frac < 0.15: sev = "medium"
            else:                  sev = "large"

            class_counts[det["class"]] += 1
            areas.append(area_frac)
            confs.append(det["confidence"])
            sev_scores.append(sev_map[sev])

        n = len(detections)
        km_per_year = km_driven / max(car_age, 1)

        features = {
            # Vehicle
            "car_age":             car_age,
            "km_driven":           km_driven,
            "km_per_year":         km_per_year,
            "purchase_price":      est_purchase_price,
            "idv":                 idv,
            "is_luxury":           int(segment == "luxury"),
            "is_premium":          int(segment in ["suv_full", "luxury"]),
            "segment_multiplier":  seg_mult,
            "depreciation_factor": dep_fac,
            "is_high_mileage":     int(km_driven > 80_000),
            "is_old_car":          int(car_age > 5),
            # Damage
            "damage_count":        n,
            "total_area_frac":     sum(areas),
            "max_area_frac":       max(areas) if areas else 0,
            "avg_area_frac":       sum(areas) / n if n else 0,
            "avg_confidence":      sum(confs) / n if n else 0,
            "min_confidence":      min(confs) if confs else 0,
            "max_severity_score":  max(sev_scores) if sev_scores else 0,
            "avg_severity_score":  sum(sev_scores) / n if n else 0,
            # Damage type counts
            **{f"count_{c.replace(' ', '_')}": class_counts[c] for c in CLASS_NAMES},
            # Binary flags
            "has_glass_shatter":   int(class_counts["glass shatter"] > 0),
            "has_crack":           int(class_counts["crack"] > 0),
            "has_lamp_broken":     int(class_counts["lamp broken"] > 0),
            "damage_value_ratio":  sum(areas) * est_purchase_price / 100_000,
        }

        vec = np.array([[features[f] for f in self.feature_names]])
        return self.scaler.transform(vec)

    def predict(
        self,
        image_path: str,
        car_model: str,
        car_age: int,
        km_driven: int,
    ) -> dict:

        detections, img_w, img_h = self._run_yolo(image_path)

        if not detections:
            return {
                "detections":      [],
                "damage_count":    0,
                "claim_amount":    0,
                "claim_min":       0,
                "claim_max":       0,
                "severity":        "No damage detected",
                "claim_recommended": False,
                "vehicle_info":    {
                    "car_model": car_model, "car_age": car_age,
                    "km_driven": km_driven, "segment": "unknown",
                },
            }

        feature_vec   = self._build_features(
            detections, img_w, img_h, car_model, car_age, km_driven
        )
        claim_amount  = max(0, float(self.model.predict(feature_vec)[0]))

        # ±20% confidence interval
        claim_min = round(claim_amount * 0.80)
        claim_max = round(claim_amount * 1.20)
        claim_amount  = round(claim_amount)

        # Severity label
        if claim_amount < 5_000:     severity, rec = "Minor",    False
        elif claim_amount < 20_000:  severity, rec = "Moderate", True
        elif claim_amount < 60_000:  severity, rec = "Severe",   True
        else:                         severity, rec = "Major",    True

        segment = resolve_segment(car_model)
        dep_fac = get_depreciation_factor(car_age)

        return {
            "detections":       detections,
            "damage_count":     len(detections),
            "claim_amount":     claim_amount,
            "claim_min":        claim_min,
            "claim_max":        claim_max,
            "severity":         severity,
            "claim_recommended": rec,
            "vehicle_info": {
                "car_model":           car_model,
                "car_age":             car_age,
                "km_driven":           km_driven,
                "segment":             segment,
                "depreciation_factor": round(dep_fac, 2),
                "idv_estimate":        round(
                    {"hatchback": 550_000, "sedan": 900_000, "suv_compact": 1_200_000,
                     "suv_full": 2_000_000, "luxury": 5_000_000}[segment] * dep_fac
                ),
            },
        }